# Databricks Lakehouse Lab: SQL and Machine Learning

This notebook is designed to run in **Databricks Community Edition**. It uses a **synthetic customer churn dataset** generated inside the notebook.

## Learning goals
By the end of the lab, students will be able to:
1. Use Databricks as a unified SQL + Python environment  
2. Explore a dataset with SQL  
3. Prepare features in Python  
4. Train and evaluate a baseline machine learning model  


In [0]:
# Databricks notebook setup
import warnings
warnings.filterwarnings("ignore")

spark.conf.set("spark.sql.shuffle.partitions", "8")
print("Spark version:", spark.version)

Spark version: 4.1.0


## Part 1 — Read the data from a csv file and store them in an SQL table

In real situation, most likely the data are already in the SQL table


In [0]:
import pandas as pd
pdf = pd.read_csv("lab_customer_churn.csv")


In [0]:
# Convert pandas DataFrame to Spark DataFrame and persist as a Delta table
df = spark.createDataFrame(pdf)
table_name = "lab_customer_churn"

spark.sql(f"DROP TABLE IF EXISTS {table_name}")

df.write.mode("overwrite").format("delta").saveAsTable(table_name)

# spark.sql(f"SELECT * FROM {table_name} LIMIT 10").show()

## Part 2 — Explore the data with SQL

Run the following SQL cells and interpret the outputs.

Show the content of table lab_customer_churn

In [0]:
%sql


customer_id,tenure,monthly_charges,support_calls,contract_type,internet_service,payment_method,senior_citizen,paperless_billing,churn
1,7,101.2,3,One year,DSL,Bank transfer,0,0,1
2,55,63.69,4,One year,Fiber optic,Mailed check,1,1,0
3,47,79.09,2,Two year,Fiber optic,Electronic check,0,1,1
4,32,18.0,1,Month-to-month,Fiber optic,Mailed check,0,0,0
5,31,41.09,0,Month-to-month,Fiber optic,Mailed check,0,0,0
6,61,62.66,3,Month-to-month,Fiber optic,Credit card,0,0,1
7,7,43.2,1,One year,Fiber optic,Mailed check,0,1,0
8,50,null,2,Month-to-month,Fiber optic,Mailed check,0,0,1
9,15,119.93,2,One year,DSL,Electronic check,0,0,1
10,7,40.58,3,One year,Fiber optic,Mailed check,1,0,1


Understand how the target `churn` is distributed

Use SELECT, COUNT and GROUP BY

In [0]:
%sql


churn_status,n_customers
0,471
1,729


For each `contract_type` find the average values of `monthly_charges`, `tenure`, `support_calls`, `churn`

In [0]:
%sql


contract_type,avg_monthly_charges,avg_tenure,avg_support_calls,churn_rate
Month-to-month,69.15,37.05,2.12,0.626
One year,69.86,34.59,2.17,0.613
Two year,66.67,36.21,2.14,0.537


Check if there are missing values in some columns (in principle, it should be done for all the columns)

Use SUM and the CASE construct: `CASE WHEN <column_name> IS NULL THEN 1 ELSE 0 END` returns 1 for nulls, 0 otherwise. 

The sum will return the number of nulls

In [0]:
%sql


missing_monthly_charges,missing_internet_service
20,20


Execute a deeper exploration

for each pair `internet_service`, `payment_method` count the number of customers and the average churn rate, order by number of customers and descending churn rate

In [0]:
%sql


internet_service,payment_method,customers,churn_rate
No,Mailed check,40,0.775
No,Credit card,44,0.682
DSL,Electronic check,100,0.65
Fiber optic,Electronic check,156,0.647
Fiber optic,Credit card,146,0.637
Fiber optic,Bank transfer,149,0.611
No,Bank transfer,41,0.61
DSL,Bank transfer,93,0.581
No,Electronic check,43,0.581
DSL,Mailed check,111,0.577


### Short discussion prompt
- Which customer groups show the highest churn?
- Which variables look potentially predictive?
- Do the missing values require treatment before modeling?

## Part 3 — Move to Python and prepare features

In [0]:
# Read the Spark table into pandas for a compact scikit-learn workflow
model_pdf = spark.table("lab_customer_churn").toPandas()

print(model_pdf.info())
display(model_pdf.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1200 entries, 0 to 1199
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   customer_id        1200 non-null   int64  
 1   tenure             1200 non-null   int64  
 2   monthly_charges    1180 non-null   float64
 3   support_calls      1200 non-null   int64  
 4   contract_type      1200 non-null   object 
 5   internet_service   1180 non-null   object 
 6   payment_method     1200 non-null   object 
 7   senior_citizen     1200 non-null   int64  
 8   paperless_billing  1200 non-null   int64  
 9   churn              1200 non-null   int64  
dtypes: float64(1), int64(6), object(3)
memory usage: 93.9+ KB
None


customer_id,tenure,monthly_charges,support_calls,contract_type,internet_service,payment_method,senior_citizen,paperless_billing,churn
1,7,101.2,3,One year,DSL,Bank transfer,0,0,1
2,55,63.69,4,One year,Fiber optic,Mailed check,1,1,0
3,47,79.09,2,Two year,Fiber optic,Electronic check,0,1,1
4,32,18.0,1,Month-to-month,Fiber optic,Mailed check,0,0,0
5,31,41.09,0,Month-to-month,Fiber optic,Mailed check,0,0,0


In [0]:
# Basic preprocessing
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

target = "churn"
feature_cols = [
    "tenure",
    "monthly_charges",
    "support_calls",
    "contract_type",
    "internet_service",
    "payment_method",
    "senior_citizen",
    "paperless_billing",
]

X = model_pdf[feature_cols]
y = model_pdf[target]

numeric_features = ["tenure", "monthly_charges", "support_calls", "senior_citizen", "paperless_billing"]
categorical_features = ["contract_type", "internet_service", "payment_method"]

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))

Training rows: 900
Test rows: 300


## Part 4 — Train and evaluate a baseline model

In [0]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report, confusion_matrix

clf = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", LogisticRegression(max_iter=1000))
])

clf.fit(X_train, y_train)
pred = clf.predict(X_test)
proba = clf.predict_proba(X_test)[:, 1]

print("Accuracy:", round(accuracy_score(y_test, pred), 3))
print("F1-score:", round(f1_score(y_test, pred), 3))
print("ROC-AUC:", round(roc_auc_score(y_test, proba), 3))
print()
print(classification_report(y_test, pred))
print("Confusion matrix:")
print(confusion_matrix(y_test, pred))

Accuracy: 0.663
F1-score: 0.765
ROC-AUC: 0.675

              precision    recall  f1-score   support

           0       0.66      0.30      0.41       118
           1       0.66      0.90      0.76       182

    accuracy                           0.66       300
   macro avg       0.66      0.60      0.59       300
weighted avg       0.66      0.66      0.62       300

Confusion matrix:
[[ 35  83]
 [ 18 164]]


## Part 5 — Execution of the experiments and output of the results

In [0]:
# Train the model
clf.fit(X_train, y_train)

# Predictions
pred = clf.predict(X_test)
proba = clf.predict_proba(X_test)[:, 1]

# Metrics
acc = accuracy_score(y_test, pred)
f1 = f1_score(y_test, pred)
auc = roc_auc_score(y_test, proba)

# Simple results presentation
print("=== Logistic Regression Results ===")
print(f"Accuracy : {acc:.3f}")
print(f"F1-score : {f1:.3f}")
print(f"ROC-AUC  : {auc:.3f}")

=== Logistic Regression Results ===
Accuracy : 0.663
F1-score : 0.765
ROC-AUC  : 0.675


## Optional extension — Compare with a Random Forest

In [0]:
from sklearn.ensemble import RandomForestClassifier

rf = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=150,
        max_depth=6,
        random_state=42
    ))
])

rf.fit(X_train, y_train)
pred_rf = rf.predict(X_test)
proba_rf = rf.predict_proba(X_test)[:, 1]

acc_rf = accuracy_score(y_test, pred_rf)
f1_rf = f1_score(y_test, pred_rf)
auc_rf = roc_auc_score(y_test, proba_rf)

print("Accuracy:", round(acc_rf, 3))
print("F1-score:", round(f1_rf, 3))
print("ROC-AUC:", round(auc_rf, 3))

Accuracy: 0.63
F1-score: 0.747
ROC-AUC: 0.635


## Final reflection
Answer these questions in 2–3 lines each:

1. What is the advantage of using SQL before model training?
2. What is the value of having SQL, Python, and MLflow in the same environment?
3. Which variables seem most useful for churn prediction?
4. What are the limitations of this lab compared with a production ML workflow?